# E13 — Replanejamento de enlaces da rede AGU-S

Este exemplo usa o diagnóstico de `e13_diagnostico.ipynb` para preparar e testar melhorias na rede instalada. Requer SDK 0.1.56 e os serviços de rede e features. O projeto de origem permanece em `projects/e10_diagnostico`; os estudos e resultados deste notebook ficam em `projects/e13_replanejamento`.

Antes de executar, confirme a reconfiguração no diagnóstico usando **Somente enlace**, sem remover rádios. A seleção é uma decisão do engenheiro: os enlaces para investigar defeito não são demandas automáticas de replanejamento. Este notebook valida a escolha salva; não confirma sugestões em nome do usuário.

O modo subárvore fica para o fluxo da E10 discutido anteriormente. Aqui preservamos todos os equipamentos e os demais enlaces, buscando novas ligações entre os mesmos extremos.

In [ ]:
from cisei_planning_sdk import PlanningClient, InstalledNetworkProject, LinkReplanningProject

client = PlanningClient("http://network-service:8080", workspace="..", timeout=1800)
source = InstalledNetworkProject("../projects/e10_diagnostico")
project = LinkReplanningProject("../projects/e13_replanejamento", source=source)

## 1. Conferir a escolha do engenheiro

O SDK lê o JSON salvo na origem, verifica a identificação do projeto e confere as remoções. A seleção deve conter apenas enlaces. Os rádios originais, inclusive repetidores e equipamentos que atendem DA/AMI, são mantidos.

In [ ]:
source.load_reconfiguration(source.path / "reconfiguration.json")

## 2. Preparar o projeto de replanejamento

A preparação calcula os custos dos enlaces instalados com a métrica calibrada da E08, reutilizando as features do diagnóstico. A topologia explícita tem um único caminho por nó: o SDK verifica que o cálculo preservou os pais instalados.

O novo scenario.toml contém somente os enlaces preservados. nodes.csv mantém os 65 rádios; os pontos que ainda alcançam o backbone ficam fixos com seus ranks acumulados. Os demais precisam recuperar a conexão, mas seus enlaces não selecionados continuam presentes. A origem permanece intacta. A preparação pode ser repetida com as mesmas entradas sem apagar os resultados.

Os 22.078 postes físicos do inventário fornecido estão em poles.csv, separados dos equipamentos. replanning.toml define o perfil de rádio proposto, altura de 7 m e um corredor de 1.500 m em torno de cada enlace. Apenas os postes desse corredor são oferecidos à busca daquele par; eles não formam uma malha global de candidatos.

In [ ]:
preparation = project.prepare(client, geo_base_url="http://features:8080")
preparation

In [ ]:
prepared_view = project.show_prepared_network()

## 3. Buscar alternativas com LinkPlanner

Para cada enlace removido, o serviço testa a ligação direta e posições candidatas para repetidores. Usa as configurações dos rádios terminais, a mesma tecnologia GE em 900 MHz e a métrica calibrada, sem mudar os enlaces instalados fora da seleção.

O custo de cada novo segmento é o maior entre seus dois sentidos. Neste exemplo, custo até 2 corresponde ao limiar de RSSI de -80 dBm na métrica configurada. A busca é limitada a cinco repetidores e três caminhos candidatos por rodada. Não é uma garantia de ótimo global nem de que haverá uma alternativa viável.

Entre os caminhos avaliados que atendem ao limite por segmento, é escolhido o de menor custo total. Essa exigência vem antes da comparação de custo com enlaces inadequados. Propostas que excedem o limite são guardadas para inspeção, mas não são incorporadas à solução consolidada. Assim, um enlace sem alternativa aceitável continua pendente. Os estudos são independentes: cada par tem seus próprios rádios propostos, mesmo quando usa o mesmo poste físico. Não se presume compartilhamento automático de repetidores entre estudos.

In [ ]:
replanning = project.replan_links(client, geo_base_url="http://features:8080")
replanning

## 4. Inspecionar os enlaces efetivamente avaliados

Os mapas mostram os grafos que foram testados, incluindo alternativas descartadas. Laranja indica os segmentos do caminho escolhido no estudo; cinza indica os outros segmentos avaliados. Clique para ver custo, tecnologia e participação na solução. A presença no caminho escolhido não implica atendimento ao limite: confira a indicação na visualização da proposta.

Nos estudos individuais, o nó azul é apenas o extremo de referência do par. Isso não transforma o rádio em um novo acesso ao backbone. Os acessos reais permanecem os originais na rede consolidada.

**GE-014 — GE-058: alternativas avaliadas e proposta escolhida.** Execute estas células se esse enlace estiver na sua seleção.

In [ ]:
candidates_014_058 = project.show_link_candidates("AGU-S-GE-014__AGU-S-GE-058")

In [ ]:
proposal_014_058 = project.show_link_solution("AGU-S-GE-014__AGU-S-GE-058")

**GE-015 — GE-066: alternativas avaliadas e proposta escolhida.** Execute estas células se esse enlace estiver na sua seleção.

In [ ]:
candidates_015_066 = project.show_link_candidates("AGU-S-GE-015__AGU-S-GE-066")

In [ ]:
proposal_015_066 = project.show_link_solution("AGU-S-GE-015__AGU-S-GE-066")

**GE-022 — GE-031: alternativas avaliadas e proposta escolhida.** Execute estas células se esse enlace estiver na sua seleção.

In [ ]:
candidates_022_031 = project.show_link_candidates("AGU-S-GE-022__AGU-S-GE-031")

In [ ]:
proposal_022_031 = project.show_link_solution("AGU-S-GE-022__AGU-S-GE-031")

**GE-024 — GE-025: alternativas avaliadas e proposta escolhida.** Execute estas células se esse enlace estiver na sua seleção.

In [ ]:
candidates_024_025 = project.show_link_candidates("AGU-S-GE-024__AGU-S-GE-025")

In [ ]:
proposal_024_025 = project.show_link_solution("AGU-S-GE-024__AGU-S-GE-025")

## 5. Inspecionar a rede consolidada

A consolidação usa apenas os enlaces preservados, com seus custos, e os caminhos aceitos. Não gera novas conexões entre os demais equipamentos. O único acesso externo continua sendo o da rede original; os pontos fixos da preparação não são tratados como novos acessos independentes nesta avaliação completa.

Enlaces laranja são novos segmentos aceitos; enlaces cinza são instalados e preservados. Nós azuis são acessos; nós vermelhos ou com contorno vermelho permanecem sem serviço; roxo identifica rádios com equipamento associado. Os componentes desconectados continuam desenhados com seus enlaces preservados. Compare esta visualização com a rede preparada e com as propostas individuais. A rede inteira pode continuar parcialmente desconectada quando uma das melhorias não atende ao limite.

In [ ]:
solution_view = project.show_network()

## 6. Arquivos do estudo

scenario.toml e nodes.csv registram o cenário após a retirada dos enlaces escolhidos. prepared_network.json guarda essa etapa. installed_network.json preserva a referência da rede instalada e seus custos; reconfiguration.json registra a decisão aplicada. features.json reúne o cache bruto reutilizado e ampliado pelas buscas.

Cada proposta e seu grafo de alternativas ficam em link_studies. replanning_result.json registra se cada estudo foi aceito. result.json e solution_scenario.toml guardam a rede consolidada e sua configuração completa, incluindo os repetidores propostos aceitos. Nada é instalado ou alterado na rede real.